# Autonomiczna jazda JetBota

Ten notebook wczytuje wytrenowany model `model.keras` (EfficientNetB0) i używa go do sterowania silnikami na podstawie obrazu z kamery w czasie rzeczywistym.

**Przepływ preprocessingu (identyczny jak w treningu):**
1. Kamera → obraz BGR `224×224`
2. `crop_image` → dolna połowa → `112×224`
3. BGR → RGB, normalizacja `/ 255.0`
4. Model → `[left_wheel, right_wheel]` → silniki

## 1. Importy i ładowanie modelu

In [3]:
import tensorflow as tf
import numpy as np
import cv2
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

# from jetbot import Robot, Camera, bgr8_to_jpeg, Heartbeat

# Ścieżka do zapisanego modelu (z main.ipynb)
MODEL_PATH = '100epochs.keras'

model = tf.keras.models.load_model(MODEL_PATH)
print('Model załadowany:', MODEL_PATH)
model.summary()

2026-05-13 11:45:35.767624: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-13 11:45:35.813094: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-13 11:45:35.813532: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model załadowany: 100epochs.keras


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 112, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 112, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 112, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 112, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 113, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 56, 112,   │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 56, 112,   │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 56, 112,   │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 56, 112,   │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 56, 112,   │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 56, 112,   │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 56, 112,   │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 56, 112,   │        512 │ block1a_se_excit

 Total params: 12,072,355 (46.05 MB)

 Trainable params: 4,010,110 (15.30 MB)

 Non-trainable params: 42,023 (164.16 KB)

 Optimizer params: 8,020,222 (30.59 MB)

## 2. Preprocessing — identyczny jak podczas treningu

In [4]:
def crop_image(img):
    imsize = img.shape
    return img[imsize[0] // 2:, :]


def preprocess(bgr_frame):
    img = crop_image(bgr_frame)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

## 3. Kamera i podgląd wideo

In [ ]:
camera = Camera.instance(width=224, height=224)

image_widget = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# Podgląd wartości predykcji
left_label    = widgets.FloatText(description='Lewy:',  layout=widgets.Layout(width='200px'))
right_label   = widgets.FloatText(description='Prawy:', layout=widgets.Layout(width='200px'))
running_label = widgets.Label(value='Status: ZATRZYMANY')

display(image_widget, widgets.HBox([left_label, right_label]), running_label)

## 4. Robot i callback predykcji

In [ ]:
robot = Robot()

def predict_and_drive(change):
    """Callback wywoływany przy każdej nowej klatce z kamery."""
    frame = change['new']
    inp   = preprocess(frame)
    left, right = model(inp, training=False).numpy()[0]

    left  = float(np.clip(left,  -1.0, 1.0))
    right = float(np.clip(right, -1.0, 1.0))

    robot.left_motor.value  = left
    robot.right_motor.value = right

    left_label.value  = round(left,  4)
    right_label.value = round(right, 4)


print('Callback predykcji gotowy.')

## 5. Zatrzymanie przy utracie połączenia (Heartbeat)

In [ ]:
def handle_heartbeat(change):
    if change['new'] == Heartbeat.Status.dead:
        stop_robot()

heartbeat = Heartbeat(period=0.5)
heartbeat.observe(handle_heartbeat, names='status')
print('Heartbeat aktywny — robot zatrzyma się przy utracie połączenia.')

## 6. START / STOP

In [ ]:
def start_robot():
    camera.observe(predict_and_drive, names='value')
    running_label.value = 'Status: JEDZIE 🚗'
    print('Autonomiczna jazda WŁĄCZONA.')

def stop_robot():
    camera.unobserve(predict_and_drive, names='value')
    robot.stop()
    running_label.value = 'Status: ZATRZYMANY 🛑'
    print('Autonomiczna jazda WYŁĄCZONA. Robot zatrzymany.')

btn_start = widgets.Button(description='▶ START', button_style='success', layout=widgets.Layout(width='150px'))
btn_stop  = widgets.Button(description='⏹ STOP',  button_style='danger',  layout=widgets.Layout(width='150px'))

btn_start.on_click(lambda _: start_robot())
btn_stop.on_click(lambda _: stop_robot())

display(widgets.HBox([btn_start, btn_stop]))

## 7. Zamknięcie kamery

Uruchom tę komórkę przed zamknięciem notebooka.

In [ ]:
stop_robot()
camera_link.unlink()
camera.stop()
print('Kamera zamknięta.')